# Presidio baseline on TAB + crapii PII data

## Start here

**What this notebook does, in one sentence:** it runs Presidio over the data, compares what Presidio redacts with the human labels, and tells you (1) how well Presidio did and (2) what in the data might need cleaning.

| 1. Configuration | Settings
| 2. Load data | Reads the three files
| 3. Data-quality checks | Looks for problems in the labels
| 4. Presidio pipeline | Runs Presidio (the slow part)
| 5. Evaluation | Scores Presidio
| 6. Error analysis | Shows Presidio's mistakes
| 7. Cleaning report | Sums up what to clean, saves CSVs

- **`tab`** = court-case documents. These are labelled in a lot of detail: every name, date, place and organisation.
- **`crapii`** = student essays. Only direct personal info is labelled, mostly student names.

Because the two are labelled so differently, results are always shown separately for each.

### Setup
These two cells load the tools the notebook needs. The first cell is only a reminder of install commands; it's commented out, so it does nothing. You already installed everything in the terminal.

In [ ]:
# Already done in the terminal. Only uncomment these if something is missing:
# !pip install presidio-analyzer pandas matplotlib tqdm
# !python -m spacy download en_core_web_lg

In [ ]:
import json, re, hashlib, random, collections, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

## 1. Configuration

- `True`: a small test run of about 240 documents
- `False`: the real run, about 7,000 documents

In [ ]:
# ---- the one setting to change ----
QUICK_TEST = True        # True = small test run (minutes). False = real run (30-90 min).
# ------------------------------------

# Finds data/processed whether this notebook is in notebooks/ or in the project root
DATA_DIR = next((p for p in [Path("../data/processed"), Path("./data/processed")] if (p / "dev.jsonl").exists()),
                Path("../data/processed"))
print("Reading data from:", DATA_DIR.resolve())
OUTPUT_DIR = Path("./presidio_eval_outputs")   # add notebooks/presidio_eval_outputs/ to .gitignore
CACHE_DIR  = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "dev", "test"]

# Fraction of each split to run through Presidio, stratified by source.
# Full train is ~19k docs (roughly 1-3 h on CPU with en_core_web_lg).
# Start small; the cache means you can raise these later without losing work.
SAMPLE_FRAC = ({"train": 0.0, "dev": 0.1, "test": 0.0} if QUICK_TEST
               else {"train": 0.10, "dev": 1.0, "test": 1.0})
SEED = 13

SPACY_MODEL     = "en_core_web_lg"
LANGUAGE        = "en"
SCORE_THRESHOLD = 0.35      # applied at evaluation time; Presidio itself runs at 0 and caches all scores
BETA            = 2.0       # F2
MATCH_THRESHOLD = 0.5       # a span "matches" if >= 50% of its non-space characters are covered
FLATTEN_GOLD    = True      # repo convention: flatten nested gold spans longest-first for entity-level metrics

# Which gold spans must be redacted. crapii spans have no identifier_type (None) and are all direct PII.
MASK_IDENTIFIER_TYPES = {"DIRECT", "QUASI", None}

# Optional extra recognizers (off = vanilla Presidio baseline). See section 4.
ADD_CUSTOM_RECOGNIZERS = False

# Tokens that don't count as "leaked" if left unredacted (honorifics, function words)
LEAK_IGNORE_TOKENS = {"mr", "mrs", "ms", "miss", "dr", "prof", "judge", "sir", "the", "of", "and",
                      "a", "an", "in", "at", "on", "de", "la", "le", "von", "van"}

## 2. Load data

This reads `train.jsonl`, `dev.jsonl` and `test.jsonl` and builds two tables: one row per **document**, and one row per **labelled span** (each piece of text a person marked as PII).

**Look at:** the small table it prints. It shows how many documents each split has, and how many of them contain PII.

In [ ]:
def load_split(split):
    path = DATA_DIR / f"{split}.jsonl"
    rows = []
    with path.open(encoding="utf-8") as f:
        for i, line in enumerate(f):
            if line.strip():
                d = json.loads(line)
                d["split"] = split          # trust the file, not the field
                d["_line"] = i
                rows.append(d)
    return rows

DOCS = {s: load_split(s) for s in SPLITS}
ALL_DOCS = [d for s in SPLITS for d in DOCS[s]]
for d in ALL_DOCS:
    d["key"] = f'{d["split"]}|{d["doc_id"]}'

def is_must_mask(span):
    return span.get("attrs", {}).get("identifier_type") in MASK_IDENTIFIER_TYPES

# Coarse label space shared by gold and predictions (enables type-aware matching)
GOLD_TO_COARSE = {
    "PERSON_PRIVATE": "PERSON", "PERSON_PUBLIC": "PERSON",
    "LOCATION_OTHER": "LOCATION", "STREET_ADDRESS": "ADDRESS",
    "DATE_OTHER": "DATE", "ORG": "ORG", "DEMOGRAPHIC": "DEMOGRAPHIC",
    "ID_NUM": "ID_NUM", "EMAIL": "EMAIL", "PHONE": "PHONE",
    "URL_PERSONAL": "URL", "OTHER_PII": "OTHER",
}

doc_df = pd.DataFrame([{
    "key": d["key"], "doc_id": d["doc_id"], "split": d["split"], "source": d["source"],
    "n_chars": len(d["text"]), "n_spans": len(d["spans"]),
    "n_must_mask": sum(is_must_mask(s) for s in d["spans"]),
} for d in ALL_DOCS])

gold_df = pd.DataFrame([{
    "key": d["key"], "split": d["split"], "source": d["source"],
    "start": s["start"], "end": s["end"], "label": s["label"],
    "coarse": GOLD_TO_COARSE.get(s["label"], "OTHER"),
    "entity_type": s.get("attrs", {}).get("entity_type"),
    "identifier_type": s.get("attrs", {}).get("identifier_type"),
    "must_mask": is_must_mask(s),
    "text": d["text"][s["start"]:s["end"]],
} for d in ALL_DOCS for s in d["spans"]])

display(doc_df.groupby(["split", "source"]).agg(docs=("key", "size"), median_chars=("n_chars", "median"),
        docs_with_pii=("n_must_mask", lambda x: int((x > 0).sum())), spans=("n_spans", "sum"),
        must_mask_spans=("n_must_mask", "sum")))

## 3. Data-quality checks (before any modelling)

this checks the **labels themselves** for problems:
- a label that starts or ends in the middle of a word
- the same name labelled in one sentence but not in the next
- the same essay appearing twice

Nothing here changes your data; it only makes a list. You'll see that list summarised in section 7, so you can skim this section.

*Technical notes:*

Each check appends rows to `issues`, which is summarised and exported at the end of the notebook.

In [ ]:
issues = []   # one row per individual finding
def flag(check, d, detail, example=""):
    issues.append({"check": check, "split": d["split"], "source": d["source"], "key": d["key"],
                   "detail": detail, "example": example[:120]})

ignored_by_key = {d["key"]: d.get("ignored_spans", []) or [] for d in ALL_DOCS}

for d in tqdm(ALL_DOCS, desc="span checks"):
    text, spans = d["text"], d["spans"]
    if not text.strip():
        flag("empty_text", d, "document text is empty")
        continue
    if "\ufffd" in text or re.search(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", text):
        flag("encoding_artifacts", d, "replacement / control characters in text")

    seen = collections.Counter()
    for s in spans:
        st, en = s["start"], s["end"]
        if not (0 <= st < en <= len(text)):
            flag("bad_offsets", d, f"{st}-{en} outside text of len {len(text)}")
            continue
        surf = text[st:en]
        if surf != surf.strip():
            flag("span_whitespace_boundary", d, s["label"], repr(surf))
        if re.match(r"^[\s.,;:()\[\]\"'“”‘’-]|.*[,;:(\[\"“‘-]$", surf, flags=re.S):
            flag("span_punct_boundary", d, s["label"], repr(surf))
        if (st > 0 and text[st - 1].isalnum() and surf[:1].isalnum()) or \
           (en < len(text) and text[en].isalnum() and surf[-1:].isalnum()):
            flag("span_cuts_word", d, s["label"], text[max(0, st - 10):en + 10])
        seen[(st, en, s["label"])] += 1
    for (st, en, lab), c in seen.items():
        if c > 1:
            flag("duplicate_span", d, f"{lab} x{c}", text[st:en])

    by_off = collections.defaultdict(set)
    for s in spans:
        by_off[(s["start"], s["end"])].add(s["label"])
    for (st, en), labs in by_off.items():
        if len(labs) > 1:
            flag("same_offsets_diff_labels", d, "/".join(sorted(labs)), text[st:en])

    srt = sorted(spans, key=lambda s: (s["start"], -s["end"]))
    for a, b in zip(srt, srt[1:]):
        if b["start"] < a["end"] and (a["start"], a["end"]) != (b["start"], b["end"]):
            kind = "nested_span" if b["end"] <= a["end"] else "partial_overlap_span"
            flag(kind, d, f'{a["label"]} / {b["label"]}', f'{text[a["start"]:a["end"]]} | {text[b["start"]:b["end"]]}')

    # label <-> attribute consistency
    for s in spans:
        a = s.get("attrs", {}) or {}
        if d["source"] == "tab" and (a.get("entity_type") is None or a.get("identifier_type") is None):
            flag("tab_missing_attrs", d, s["label"], text[s["start"]:s["end"]])
        if s["label"] == "PERSON_PUBLIC" and a.get("identifier_type") in {"DIRECT", "QUASI"}:
            flag("public_person_marked_maskable", d, a.get("identifier_type"), text[s["start"]:s["end"]])

**Unlabelled repeat mentions.** For each document, look for other occurrences of strings that are already annotated as must-mask PII, where that occurrence isn't covered by any gold or ignored span. These are the most likely *missing annotations*, which is a classic cleaning target, especially for names in crapii.

In [ ]:
def covered_mask(n, spans):
    m = np.zeros(n, dtype=bool)
    for s in spans:
        m[max(0, s["start"]):min(n, s["end"])] = True
    return m

for d in tqdm(ALL_DOCS, desc="missed mentions"):
    text = d["text"]
    if not d["spans"]:
        continue
    cov = covered_mask(len(text), d["spans"] + ignored_by_key[d["key"]])
    surfaces = {(text[s["start"]:s["end"]], s["label"]) for s in d["spans"]
                if is_must_mask(s) and len(text[s["start"]:s["end"]].strip()) >= 3
                and s["label"] not in {"DATE_OTHER"}}
    for surf, lab in surfaces:
        pat = r"(?<!\w)" + re.escape(surf) + r"(?!\w)"
        for m in re.finditer(pat, text):
            if not cov[m.start():m.end()].any():
                flag("unlabelled_repeat_mention", d, lab,
                     text[max(0, m.start() - 30):m.end() + 30].replace("\n", " "))

**Near-duplicate documents** within or across splits. The build step dedups by exact text hash (keeping test > dev > train). This check hashes *normalised* text (lower-cased, whitespace collapsed), so it catches copies that differ only in spacing or line breaks, which the exact hash misses.

Duplicates across splits mean train/test contamination, which inflates scores for any trained model such as GLiNER in Phase 2.

In [ ]:
def norm_hash(t):
    return hashlib.md5(re.sub(r"\s+", " ", t.lower()).strip().encode()).hexdigest()

doc_df["text_hash"] = [norm_hash(d["text"]) for d in ALL_DOCS]
dup_groups = doc_df.groupby("text_hash").filter(lambda g: len(g) > 1)
for h, g in dup_groups.groupby("text_hash"):
    kind = "near_duplicate_doc_cross_split" if g["split"].nunique() > 1 else "near_duplicate_doc_within_split"
    for _, r in g.iterrows():
        issues.append({"check": kind, "split": r.split, "source": r.source, "key": r.key,
                       "detail": f"group of {len(g)}: " + ", ".join(g["key"]), "example": ""})

dup_ids = doc_df[doc_df.duplicated(["split", "doc_id"], keep=False)]
for _, r in dup_ids.iterrows():
    issues.append({"check": "duplicate_doc_id", "split": r.split, "source": r.source, "key": r.key,
                   "detail": "doc_id repeated within split", "example": ""})

**Label consistency across the corpus.** These are surface strings that receive different labels, or different masking decisions, in different places. Some are legitimate (e.g. "Denmark" as LOCATION vs as part of an ORG), but large groups usually indicate guideline drift between annotators.

In [ ]:
g = gold_df.assign(norm=gold_df["text"].str.lower().str.strip())
g = g[g["norm"].str.len() >= 3]
conflicts = (g.groupby(["source", "norm"])
               .agg(n=("label", "size"), labels=("label", lambda x: dict(collections.Counter(x))),
                    mask_decisions=("must_mask", lambda x: dict(collections.Counter(x))),
                    n_labels=("label", "nunique"), n_mask=("must_mask", "nunique"))
               .query("n >= 5 and (n_labels > 1 or n_mask > 1)")
               .sort_values("n", ascending=False))
print(f"{len(conflicts)} surface strings with conflicting label or masking decisions (n >= 5)")
display(conflicts.head(25))

In [ ]:
print("TAB label x identifier_type")
display(pd.crosstab(gold_df.query("source=='tab'")["label"], gold_df.query("source=='tab'")["identifier_type"], margins=True))

print("Label distribution per split (% of spans) - large shifts between splits are a red flag")
display((pd.crosstab([gold_df["source"], gold_df["label"]], gold_df["split"], normalize="columns") * 100).round(2))

In [ ]:
issues_df = pd.DataFrame(issues, columns=["check", "split", "source", "key", "detail", "example"])
quality_summary = (issues_df.groupby(["check", "source"]).agg(findings=("key", "size"), docs=("key", "nunique"))
                   .reset_index().sort_values("findings", ascending=False))
display(quality_summary)
for chk in quality_summary["check"].unique()[:8]:
    print(f"\n--- {chk} (examples)")
    display(issues_df[issues_df.check == chk].sample(min(5, (issues_df.check == chk).sum()), random_state=SEED)
            [["source", "key", "detail", "example"]])

## 4. Presidio pipeline

This sets up Presidio and runs it over every document. **This is the slow part.**

Results are saved as they go, so if you stop it or close VS Code, the next run picks up where it left off. You never pay for the same document twice.

**Look at:** the list of entity types it prints. Check that `ORGANIZATION` is in it.

*Technical notes:*

The spaCy -> Presidio entity mapping is set explicitly so the run is reproducible across Presidio versions.

By default Presidio multiplies spaCy `ORG` scores by 0.4, which puts them at about 0.34, below most thresholds. Here that down-weighting is switched off, and you control it through `SCORE_THRESHOLD` instead. Check the printed list of supported entities to confirm `ORGANIZATION` is present in your version.

In [ ]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider

nlp_config = {
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": LANGUAGE, "model_name": SPACY_MODEL}],
    "ner_model_configuration": {
        "model_to_presidio_entity_mapping": {
            "PER": "PERSON", "PERSON": "PERSON", "NORP": "NRP",
            "FAC": "LOCATION", "LOC": "LOCATION", "GPE": "LOCATION", "LOCATION": "LOCATION",
            "ORG": "ORGANIZATION", "ORGANIZATION": "ORGANIZATION",
            "DATE": "DATE_TIME", "TIME": "DATE_TIME",
        },
        "low_confidence_score_multiplier": 0.4,
        "low_score_entity_names": [],
        "labels_to_ignore": ["CARDINAL", "EVENT", "LANGUAGE", "LAW", "MONEY",
                             "ORDINAL", "PERCENT", "PRODUCT", "QUANTITY", "WORK_OF_ART"],
    },
}
nlp_engine = NlpEngineProvider(nlp_configuration=nlp_config).create_engine()
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, supported_languages=[LANGUAGE])

if ADD_CUSTOM_RECOGNIZERS:
    # Example: ECHR application numbers ("no. 36244/06") - vanilla Presidio has nothing for these
    analyzer.registry.add_recognizer(PatternRecognizer(
        supported_entity="ECHR_APP_NO",
        patterns=[Pattern("echr_app_no", r"\b\d{1,6}/\d{2}\b", 0.6)],
        context=["application", "applications", "no", "nos"]))
    # Example: social handles like "@zzadkth"
    analyzer.registry.add_recognizer(PatternRecognizer(
        supported_entity="SOCIAL_HANDLE", patterns=[Pattern("handle", r"(?<!\w)@[A-Za-z0-9_]{3,30}\b", 0.5)]))

print(sorted(analyzer.get_supported_entities()))

In [ ]:
PRESIDIO_TO_COARSE = {
    "PERSON": "PERSON", "LOCATION": "LOCATION", "DATE_TIME": "DATE", "NRP": "DEMOGRAPHIC",
    "ORGANIZATION": "ORG", "EMAIL_ADDRESS": "EMAIL", "PHONE_NUMBER": "PHONE", "URL": "URL",
    "ECHR_APP_NO": "ID_NUM", "SOCIAL_HANDLE": "ID_NUM",
}
_ID_HINTS = ("SSN", "NUMBER", "LICENSE", "PASSPORT", "CARD", "IBAN", "CRYPTO", "NHS", "ITIN",
             "IP_ADDRESS", "CODE", "_ID", "AADHAAR", "PAN", "NRIC", "ABN", "ACN", "TFN", "MEDICARE")
def to_coarse(entity_type):
    if entity_type in PRESIDIO_TO_COARSE:
        return PRESIDIO_TO_COARSE[entity_type]
    return "ID_NUM" if any(h in entity_type for h in _ID_HINTS) else "OTHER"

print({e: to_coarse(e) for e in sorted(analyzer.get_supported_entities())})

In [ ]:
# Pluggable predictor: any function text -> [{"start", "end", "entity_type", "score"}].
# Swap this for GLiNER in Phase 2 (and extend PRESIDIO_TO_COARSE / to_coarse with its labels).
PREDICTOR_NAME = "presidio"

def predict_fn(text):
    res = analyzer.analyze(text=text, language=LANGUAGE, score_threshold=0.0)
    return [{"start": r.start, "end": r.end, "entity_type": r.entity_type, "score": round(float(r.score), 4)} for r in res]

### Run (cached)

In [ ]:
def sample_docs():
    rng = random.Random(SEED)
    out = []
    for split in SPLITS:
        frac = SAMPLE_FRAC.get(split, 0)
        if frac <= 0:
            continue
        by_src = collections.defaultdict(list)
        for d in DOCS[split]:
            by_src[d["source"]].append(d)
        for src, lst in sorted(by_src.items()):
            k = len(lst) if frac >= 1 else max(1, round(len(lst) * frac))
            out.extend(rng.sample(lst, k) if k < len(lst) else lst)
    return out

cfg_tag = hashlib.md5(json.dumps([PREDICTOR_NAME, SPACY_MODEL, nlp_config, ADD_CUSTOM_RECOGNIZERS], sort_keys=True, default=str).encode()).hexdigest()[:8]
CACHE_PATH = CACHE_DIR / f"{PREDICTOR_NAME}_preds_{cfg_tag}.jsonl"

def load_cache(path):
    cache = {}
    if path.exists():
        with path.open(encoding="utf-8") as f:
            for line in f:
                r = json.loads(line); cache[r["key"]] = r["preds"]
    return cache

def run_predictor(docs, cache_path=CACHE_PATH):
    cache = load_cache(cache_path)
    todo = [d for d in docs if d["key"] not in cache]
    print(f"{len(docs)} docs requested, {len(docs) - len(todo)} cached, {len(todo)} to run -> {cache_path.name}")
    failures = []
    t0 = time.time()
    with cache_path.open("a", encoding="utf-8") as f:
        for d in tqdm(todo, desc=PREDICTOR_NAME):
            try:
                preds = predict_fn(d["text"])
            except Exception as e:                       # keep going; record the failure
                failures.append((d["key"], repr(e))); continue
            cache[d["key"]] = preds
            f.write(json.dumps({"key": d["key"], "preds": preds}) + "\n")
    if todo:
        print(f"ran {len(todo)} docs in {time.time() - t0:.0f}s; failures: {len(failures)}")
    return {d["key"]: cache[d["key"]] for d in docs if d["key"] in cache}, failures

EVAL_DOCS = sample_docs()
PREDS, FAILURES = run_predictor(EVAL_DOCS)
EVAL_DOCS = [d for d in EVAL_DOCS if d["key"] in PREDS]
display(pd.Series(collections.Counter((d["split"], d["source"]) for d in EVAL_DOCS), name="docs evaluated"))

## 5. Evaluation

This compares what Presidio redacted with the human labels and gives scores. The big table has one row per split and source. The five columns that matter most:

| Column | Question it answers | Good is... |
|---|---|---|
| `span_F2` | Overall score, weighted towards catching PII | high |
| `span_recall` | Of all the real PII, how much did Presidio catch? | high |
| `span_precision` | Of what Presidio redacted, how much was really PII? | high |
| `over_redaction_rate` | How much normal text got blacked out by mistake? | low |
| `leak_rate_span` | How much real PII was left visible? | low ⚠️ the privacy one |

You'll see each row twice: **`raw`** counts every Presidio guess, and **`scoped`** ignores guesses of types a source never labels, which is fairer. **Read the `scoped` rows.**

*Technical notes on how things are counted (optional):*

Scoring rules:

- **Character-level metrics** use non-whitespace characters only. Characters inside TAB `ignored_spans` (MISC / QUANTITY) are excluded entirely: redacting them is neither rewarded nor penalised.
- **Nested gold spans** are flattened longest-first for entity-level metrics (`FLATTEN_GOLD`), so a nested name inside an ORG isn't counted twice. Character-level metrics use the union of all spans, so nesting doesn't matter there.
- **A gold span is detected** if at least `MATCH_THRESHOLD` of its characters are redacted by *any* prediction. It is detected **type-aware** if that coverage comes from predictions of the same coarse type.
- **A prediction is a true positive** if at least `MATCH_THRESHOLD` of its (non-ignored) characters fall on must-mask gold.
- **A gold span leaks** if any of its meaningful word tokens (honorifics and function words excluded) is not fully redacted. This is stricter than "detected": for example, `Mr Henrik [REDACTED]` counts as detected but leaked.
- **Over-redaction** is the share of non-PII characters that got redacted. For TAB we also report how often spans explicitly marked `NO_MASK` were redacted, and separately how often `PERSON_PUBLIC` names (judges, public figures) were. That last one is the "historical figure" over-redaction case from the project spec.

In [ ]:
SCOPE = {src: set(gold_df.loc[gold_df.source == src, "coarse"]) for src in gold_df.source.unique()}
print("scoped view keeps predicted types:", {k: sorted(v) for k, v in SCOPE.items()})

DOC_BY_KEY = {d["key"]: d for d in ALL_DOCS}
_TOK = re.compile(r"\w+", flags=re.U)

def filter_preds(preds, source, threshold, view):
    out = []
    for p in preds:
        if p["score"] < threshold:
            continue
        c = to_coarse(p["entity_type"])
        if view == "scoped" and c not in SCOPE.get(source, set()):
            continue
        out.append({**p, "coarse": c})
    return out

def flatten_longest_first(spans):
    """Repo convention for nested TAB spans: keep the longest span, drop any span overlapping a kept one."""
    kept = []
    for s in sorted(spans, key=lambda s: (-(s["end"] - s["start"]), s["start"])):
        if all(s["end"] <= k["start"] or s["start"] >= k["end"] for k in kept):
            kept.append(s)
    return sorted(kept, key=lambda s: s["start"])

def evaluate_doc(d, preds):
    text, n = d["text"], len(d["text"])
    nonspace = np.fromiter((not ch.isspace() for ch in text), dtype=bool, count=n)
    must = [s for s in d["spans"] if is_must_mask(s)]
    nomask = [s for s in d["spans"] if not is_must_mask(s)]
    must_m = covered_mask(n, must) & nonspace
    ign_m = covered_mask(n, ignored_by_key[d["key"]]) & nonspace & ~must_m
    pred_m = covered_mask(n, preds) & nonspace
    nomask_m = covered_mask(n, nomask)
    typed = collections.defaultdict(lambda: np.zeros(n, dtype=bool))
    for p in preds:
        typed[p["coarse"]][p["start"]:p["end"]] = True

    clean_m = nonspace & ~must_m & ~ign_m
    doc = {"key": d["key"], "split": d["split"], "source": d["source"],
           "char_tp": int((pred_m & must_m).sum()), "char_fp": int((pred_m & clean_m).sum()),
           "char_fn": int((must_m & ~pred_m).sum()), "clean_chars": int(clean_m.sum()),
           "n_must": len(must), "n_pred": len(preds)}

    gold_rows = []
    for s in (flatten_longest_first(must) if FLATTEN_GOLD else must):
        st, en = s["start"], s["end"]
        ns = nonspace[st:en]
        denom = max(1, int(ns.sum()))
        cov = (pred_m[st:en] & ns).sum() / denom
        cov_t = (typed[GOLD_TO_COARSE.get(s["label"], "OTHER")][st:en] & ns).sum() / denom
        toks = [m for m in _TOK.finditer(text, st, en) if m.group().lower() not in LEAK_IGNORE_TOKENS] \
               or list(_TOK.finditer(text, st, en))
        leaked = [m.group() for m in toks if not pred_m[m.start():m.end()].all()]
        if not toks:
            leaked = [text[st:en]] if cov < 1 else []
        gold_rows.append({"key": d["key"], "split": d["split"], "source": d["source"], "label": s["label"],
                          "identifier_type": s.get("attrs", {}).get("identifier_type"),
                          "text": text[st:en], "coverage": cov, "detected": cov >= MATCH_THRESHOLD,
                          "detected_typed": cov_t >= MATCH_THRESHOLD, "n_tokens": max(1, len(toks)),
                          "n_leaked_tokens": len(leaked), "leaked": bool(leaked)})

    pred_rows = []
    for p in preds:
        st, en = p["start"], p["end"]
        scorable = nonspace[st:en] & ~ign_m[st:en]
        if not scorable.any():
            continue                                    # prediction sits entirely in ignored spans
        on_pii = (must_m[st:en] & scorable).sum() / scorable.sum()
        pred_rows.append({"key": d["key"], "split": d["split"], "source": d["source"],
                          "start": st, "end": en, "entity_type": p["entity_type"], "coarse": p["coarse"], "score": p["score"],
                          "text": text[st:en], "tp": on_pii >= MATCH_THRESHOLD,
                          "on_nomask": bool(nomask_m[st:en].any())})

    nomask_rows = [{"key": d["key"], "split": d["split"], "source": d["source"], "label": s["label"],
                    "redacted": (pred_m[s["start"]:s["end"]] & nonspace[s["start"]:s["end"]]).sum()
                                / max(1, nonspace[s["start"]:s["end"]].sum()) >= MATCH_THRESHOLD}
                   for s in nomask]
    doc["any_leak"] = any(g["leaked"] for g in gold_rows)
    doc["any_direct_leak"] = any(g["leaked"] and g["identifier_type"] in ("DIRECT", None) for g in gold_rows)
    return doc, gold_rows, pred_rows, nomask_rows

def evaluate(docs, threshold=SCORE_THRESHOLD, views=("raw", "scoped")):
    out = {k: [] for k in ("docs", "gold", "pred", "nomask")}
    for view in views:
        for d in docs:
            preds = filter_preds(PREDS[d["key"]], d["source"], threshold, view)
            doc, g, p, nm = evaluate_doc(d, preds)
            for name, rows in (("docs", [doc]), ("gold", g), ("pred", p), ("nomask", nm)):
                out[name].extend({**r, "view": view} for r in rows)
    return {k: pd.DataFrame(v) for k, v in out.items()}

def fbeta(p, r, beta=BETA):
    return 0.0 if p + r == 0 else (1 + beta**2) * p * r / (beta**2 * p + r)

def summarise(R, by=("view", "split", "source")):
    by = list(by)
    rows = []
    for keys, dg in R["docs"].groupby(by):
        sel = dict(zip(by, keys if isinstance(keys, tuple) else (keys,)))
        q = lambda df: df.loc[np.logical_and.reduce([df[k] == v for k, v in sel.items()])] if len(df) else df
        g, p, nm = q(R["gold"]), q(R["pred"]), q(R["nomask"])
        span_r = g["detected"].mean() if len(g) else np.nan
        span_p = p["tp"].mean() if len(p) else np.nan
        tp, fp, fn = dg.char_tp.sum(), dg.char_fp.sum(), dg.char_fn.sum()
        char_p = tp / (tp + fp) if tp + fp else np.nan
        char_r = tp / (tp + fn) if tp + fn else np.nan
        has = dg[dg.n_must > 0]; clean = dg[dg.n_must == 0]
        direct = g[(g.identifier_type == "DIRECT") | g.identifier_type.isna()] if len(g) else g
        rows.append({**sel, "docs": len(dg), "gold_spans": len(g), "pred_spans": len(p),
            "span_precision": span_p, "span_recall": span_r,
            "span_F2": fbeta(span_p, span_r) if len(g) and len(p) else np.nan,
            "char_precision": char_p, "char_recall": char_r,
            "char_F2": fbeta(char_p, char_r) if not (np.isnan(char_p) or np.isnan(char_r)) else np.nan,
            "over_redaction_rate": fp / dg.clean_chars.sum() if dg.clean_chars.sum() else np.nan,
            "nomask_redaction_rate": nm["redacted"].mean() if len(nm) else np.nan,
            "public_person_redaction_rate": nm.loc[nm.label == "PERSON_PUBLIC", "redacted"].mean()
                                            if len(nm) and (nm.label == "PERSON_PUBLIC").any() else np.nan,
            "false_alarm_doc_rate": (clean.n_pred > 0).mean() if len(clean) else np.nan,
            "leak_rate_span": g["leaked"].mean() if len(g) else np.nan,
            "leak_rate_token": g.n_leaked_tokens.sum() / g.n_tokens.sum() if len(g) else np.nan,
            "leak_rate_direct": direct["leaked"].mean() if len(direct) else np.nan,
            "leak_rate_doc": has["any_leak"].mean() if len(has) else np.nan,
            "direct_leak_rate_doc": has["any_direct_leak"].mean() if len(has) else np.nan})
    return pd.DataFrame(rows)

R = evaluate(EVAL_DOCS)
summary = summarise(R)
pct_cols = [c for c in summary.columns if c not in ("view", "split", "source", "docs", "gold_spans", "pred_spans")]
display(summary.style.format({c: "{:.1%}" for c in pct_cols}, na_rep="-"))

### Recall and leakage by PII type

🧭 **In plain words:** the same scores, split by kind of PII (names, dates, IDs...). This shows *which kinds* Presidio is good or bad at. Expect ID numbers in `tab` to score near zero, because Presidio has no rule for court case numbers.

In [ ]:
def recall_by_type(R, view="scoped", extra=()):
    g = R["gold"][R["gold"].view == view]
    keys = ["source", "label", *extra]
    t = (g.groupby(keys).agg(n=("detected", "size"), recall=("detected", "mean"),
                             recall_type_aware=("detected_typed", "mean"), leak_rate=("leaked", "mean"),
                             mean_coverage=("coverage", "mean"))
          .sort_values(["source", "n"], ascending=[True, False]))
    return t

rbt = recall_by_type(R)
display(rbt.style.format({"recall": "{:.1%}", "recall_type_aware": "{:.1%}", "leak_rate": "{:.1%}", "mean_coverage": "{:.1%}"}))

print("TAB, by label x identifier_type")
display(recall_by_type(R, extra=("identifier_type",)).loc["tab"]
        .style.format({"recall": "{:.1%}", "recall_type_aware": "{:.1%}", "leak_rate": "{:.1%}", "mean_coverage": "{:.1%}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, src in zip(axes, ["tab", "crapii"]):
    if src not in rbt.index.get_level_values(0):
        continue
    t = rbt.loc[src]
    x = np.arange(len(t)); w = 0.38
    ax.bar(x - w/2, t["recall"], w, label="recall (any type)")
    ax.bar(x + w/2, t["leak_rate"], w, label="leak rate")
    ax.set_xticks(x); ax.set_xticklabels([f"{l}\n(n={n})" for l, n in zip(t.index, t.n)], rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1); ax.set_title(f"{src} - scoped view"); ax.legend(fontsize=8); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.show()

### Over-redaction of spans that should stay visible (TAB `NO_MASK`)
in court cases, some names (like judges) are *public* and should stay visible. This shows how often Presidio blacked them out anyway.

This is the case the Phase 2 LLM verifier is meant to send back for correction: e.g. `PERSON_PUBLIC` (judges, public figures), country names, and courts.

In [ ]:
nm = R["nomask"][R["nomask"].view == "scoped"]
if len(nm):
    display(nm.groupby(["source", "label"]).agg(n=("redacted", "size"), redaction_rate=("redacted", "mean"))
              .sort_values("n", ascending=False).style.format({"redaction_rate": "{:.1%}"}))

### Precision by predicted entity type
which of Presidio's detectors make the most mistakes.
Which Presidio recognizers are producing the false positives?

In [ ]:
p = R["pred"][R["pred"].view == "raw"]
prec = (p.groupby(["source", "entity_type"]).agg(n=("tp", "size"), precision=("tp", "mean"),
         hits_nomask=("on_nomask", "mean"), median_score=("score", "median"))
         .sort_values(["source", "n"], ascending=[True, False]))
display(prec.style.format({"precision": "{:.1%}", "hits_nomask": "{:.1%}", "median_score": "{:.2f}"}))

### Score-threshold sweep (dev only, scoped view)
🧭 **In plain words:** Presidio gives each guess a confidence score. This tries different cut-offs ("only redact if confidence is above X") and finds the best one. The chart shows the trade-off: a lower cut-off means less leakage but more over-redaction. The test results at the end use the best cut-off.

Tune on **dev**, then report **test** at the chosen threshold. Never tune on test.

In [ ]:
tune_docs = [d for d in EVAL_DOCS if d["split"] == "dev"] or EVAL_DOCS
sweep = []
for th in tqdm(np.round(np.arange(0.0, 1.0, 0.05), 2), desc="threshold sweep"):
    s = summarise(evaluate(tune_docs, threshold=th, views=("scoped",)), by=("source",))
    sweep.append(s.assign(threshold=th))
sweep = pd.concat(sweep, ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (src, t) in zip(axes, sweep.groupby("source")):
    for col in ["span_precision", "span_recall", "span_F2", "char_F2", "leak_rate_span", "over_redaction_rate"]:
        ax.plot(t.threshold, t[col], marker=".", label=col)
    best = t.loc[t.span_F2.idxmax()]
    ax.axvline(best.threshold, ls="--", c="grey")
    ax.set_title(f"{src}: best span F2 {best.span_F2:.3f} @ {best.threshold}"); ax.set_xlabel("score threshold")
    ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

best_th = sweep.loc[sweep.groupby("source").span_F2.idxmax(), ["source", "threshold", "span_F2", "char_F2", "leak_rate_span"]]
display(best_th)

In [ ]:
# Test results at the dev-tuned threshold, per source
test_docs = [d for d in EVAL_DOCS if d["split"] == "test"]
if test_docs:
    tuned = []
    for _, b in best_th.iterrows():
        docs_src = [d for d in test_docs if d["source"] == b.source]
        tuned.append(summarise(evaluate(docs_src, threshold=b.threshold, views=("scoped",))).assign(threshold=b.threshold))
    tuned = pd.concat(tuned, ignore_index=True)
    display(tuned.style.format({c: "{:.1%}" for c in pct_cols}, na_rep="-"))

## 6. Error analysis -> cleaning candidates

This lists Presidio's most common mistakes:
- **missed PII** (things that leaked)
- **false alarms** (things it redacted that weren't PII)

Some "false alarms" may actually be **missing labels in the data**, where Presidio was right and the human labeller missed something. The last table here collects those for you to review.

*Technical notes:*

In the error lists, model errors and annotation errors are mixed together, and this section is about telling them apart. Two signals point to annotation problems rather than Presidio problems:

1. A high-confidence prediction that looks exactly like PII, but where gold says it isn't.
2. A false-positive string that is labelled as PII elsewhere in the same document.

Two crapii caveats:

- CRAPII labels only **student** names. Names of instructors, authors and quoted speakers are intentionally unlabelled; in this release they're also replaced by surrogates, so they look like real private names. Treat flagged crapii `PERSON` hits as "over-redaction vs. missing label" decisions to make against `configs/labels.yaml`, not automatic fixes.
- Similarly, crapii's `URL_PERSONAL` covers only *personal* URLs, so Presidio hits on course links (coursera, youtube) are usually correct false positives. They are kept at the bottom of the review list.

In [ ]:
g = R["gold"][R["gold"].view == "scoped"]
print("Most frequently leaked gold strings (false negatives)")
display(g[g.leaked].groupby(["source", "label", "text"]).size().rename("times_leaked")
        .sort_values(ascending=False).groupby(level=[0, 1]).head(8).to_frame())

p = R["pred"][R["pred"].view == "scoped"]
print("Most frequent false-positive strings")
display(p[~p.tp].groupby(["source", "entity_type", "text"]).size().rename("times")
        .sort_values(ascending=False).groupby(level=[0, 1]).head(8).to_frame())

In [ ]:
gold_surfaces = gold_df[gold_df.must_mask].groupby("key")["text"].apply(lambda s: set(s.str.lower()))
fp = p[~p.tp].copy()
fp["labelled_elsewhere_in_doc"] = [t.lower() in gold_surfaces.get(k, set()) for k, t in zip(fp.key, fp.text)]
name_like = re.compile(r"^[A-Z][a-z'’-]+(?: [A-Z][a-z'’-]+){1,3}$")
fp["name_like"] = fp.text.map(lambda t: bool(name_like.match(t.strip())))

# Strings that are false positives in many documents ("Mind Mapping", "Human Rights") are systematic
# model errors, not annotation gaps, so they are excluded from the review list.
fp_doc_freq = fp.groupby(fp.text.str.lower())["key"].nunique()
fp["fp_doc_freq"] = fp.text.str.lower().map(fp_doc_freq)
MAX_FP_DOC_FREQ = 3

annotation_candidates = fp[
    (fp.fp_doc_freq <= MAX_FP_DOC_FREQ) & (
        fp.labelled_elsewhere_in_doc
        | ((fp.source == "crapii") & (fp.coarse == "PERSON") & fp.name_like & (fp.score >= 0.85))
        | ((fp.source == "crapii") & fp.coarse.isin(["EMAIL", "PHONE", "URL", "ID_NUM"]) & (fp.score >= 0.5)))
].copy()
# review order: same string already labelled in the doc > contact details > names > URLs
annotation_candidates["priority"] = np.select(
    [annotation_candidates.labelled_elsewhere_in_doc, annotation_candidates.coarse.isin(["EMAIL", "PHONE", "ID_NUM"]),
     annotation_candidates.coarse == "PERSON"], [0, 1, 2], 3)
annotation_candidates = annotation_candidates.sort_values(["priority", "score"], ascending=[True, False])
print(f"{len(annotation_candidates)} predictions that may be MISSING gold annotations")
display(annotation_candidates[["source", "key", "entity_type", "score", "text", "labelled_elsewhere_in_doc", "fp_doc_freq"]].head(30))

In [ ]:
def context(key, start, end, width=60):
    t = DOC_BY_KEY[key]["text"]
    return (t[max(0, start - width):start] + "[[" + t[start:end] + "]]" + t[end:end + width]).replace("\n", " ")

annotation_candidates = annotation_candidates.assign(
    context=[context(k, a, b) for k, a, b in zip(annotation_candidates.key, annotation_candidates.start, annotation_candidates.end)])
for _, r in annotation_candidates.head(10).iterrows():
    print(f"[{r.source} {r.key} {r.entity_type} {r.score:.2f}] ...{r.context}...")

## 7. Cleaning report & exports

This answers **"does the data need cleaning?"** You get one table: each type of problem, how many documents it affects, and a suggested fix. Everything is also saved as CSV files in `presidio_eval_outputs/`, so you can open the results in Excel.

In [ ]:
for _, r in annotation_candidates.iterrows():
    issues.append({"check": "possible_missing_annotation (model-flagged)", "split": r.split, "source": r.source,
                   "key": r.key, "detail": f"{r.entity_type} score={r.score:.2f}", "example": r.context[:120]})
issues_df = pd.DataFrame(issues)

SUGGESTED_ACTION = {
    "bad_offsets": "Fix or drop - spans cannot be scored.",
    "empty_text": "Drop document.",
    "encoding_artifacts": "Re-decode source text; check offsets after.",
    "span_whitespace_boundary": "Strip span and shift offsets.",
    "span_punct_boundary": "Trim punctuation, or accept; inflates exact-match FNs only.",
    "span_cuts_word": "Inspect - usually a tokenisation bug in conversion.",
    "duplicate_span": "Deduplicate.",
    "same_offsets_diff_labels": "Pick one label per guideline.",
    "nested_span": "Expected in TAB (repo flattens longest-first); only a problem for BIO/GLiNER training data.",
    "partial_overlap_span": "Resolve manually; longest-first flattening keeps only one of the pair.",
    "tab_missing_attrs": "Fill identifier_type or exclude from masking metrics.",
    "public_person_marked_maskable": "Check against TAB guidelines.",
    "unlabelled_repeat_mention": "Likely missing annotation - add span (propagate within doc).",
    "near_duplicate_doc_cross_split": "Normalise whitespace/case before hashing in src.ingest.build dedup (keeps test > dev > train).",
    "near_duplicate_doc_within_split": "Same fix in src.ingest.build dedup.",
    "duplicate_doc_id": "Make ids unique.",
    "possible_missing_annotation (model-flagged)": "Manual review; add span if PII.",
}
cleaning_report = (issues_df.groupby(["check", "source"]).agg(findings=("key", "size"), docs_affected=("key", "nunique"))
                   .reset_index())
cleaning_report["suggested_action"] = cleaning_report["check"].map(SUGGESTED_ACTION)
cleaning_report = cleaning_report.sort_values("findings", ascending=False)
display(cleaning_report)

In [ ]:
summary.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
rbt.to_csv(OUTPUT_DIR / "recall_by_type.csv")
prec.to_csv(OUTPUT_DIR / "precision_by_entity.csv")
sweep.to_csv(OUTPUT_DIR / "threshold_sweep_dev.csv", index=False)
issues_df.to_csv(OUTPUT_DIR / "data_issues_detail.csv", index=False)
cleaning_report.to_csv(OUTPUT_DIR / "cleaning_report.csv", index=False)
annotation_candidates.to_csv(OUTPUT_DIR / "annotation_review_candidates.csv", index=False)
R["gold"].to_csv(OUTPUT_DIR / "gold_span_results.csv", index=False)
R["pred"].to_csv(OUTPUT_DIR / "pred_span_results.csv", index=False)
print("wrote:", *sorted(p.name for p in OUTPUT_DIR.glob("*.csv")), sep="\n  ")